# **Tutorial per l'esplorazione dei dati di navigazione di Fermi-LAT**

#### **Tutorial a cura di L. Di Venere, P. Cristarella Orestano**
**con contributi di D. Serini, E. Bissaldi, F. de Palma, F. Gargano, A. Adelfio**
#### Contatti: [email](mailto:astroparticellebari@gmail.com)


Questo tutorial mostra come esplorare i dati di navigazione del satellite Fermi-LAT, indicati solitamente con il nome "*FT2*".

#### ISTRUZIONI

La piattaforma che stiamo utilizzando si chima Google Colab, che fornisce un *python-notebook*, cioè una interfaccia che ci consente di eseguire comandi nel linguaggio di programmazione [python](https://it.wikipedia.org/wiki/Python)  e visualizzare dati e grafici.

Per utilizzare il *notebook*: ogni blocco di comandi può essere eseguito cliccando sul tasto "play" situato in alto a sinistra della cella, oppure ponendo il cursore nella cella e cliccando Shift+Invio oppure Ctrl+Invio.
All'esecuzione del primo blocco comparirà una finestra di pop-up, nella quale bisogna cliccare sul tasto "Run anyway".

Il *notebook* può essere modificato, ma le modifiche non verranno salvate. Per salvare le proprie modifiche è necessario salvare una copia del *notebook* nel proprio account GoogleDrive, cliccando sul tasto "Copy to Drive", posizionato in alto sotto la barra dei Menù.

Importiamo le librerie necessarie alla lettura e rappresentazione dei dati

In [ ]:
%matplotlib inline
import os,sys
import time, datetime
import numpy as np
import matplotlib.pyplot as plt
import astropy.io.fits as pyfits
from astropy.table import Table
from astropy import units as u
from astropy.coordinates import SkyCoord, search_around_sky, get_sun
from astropy.visualization.wcsaxes import SphericalCircle
from astropy.time import Time

from matplotlib import patheffects
from matplotlib.colors import LogNorm
from matplotlib.dates import DateFormatter
from matplotlib.offsetbox import TextArea, DrawingArea, OffsetImage,AnnotationBbox
from IPython.display import display, clear_output, Image
from IPython.utils.text import columnize
from ipywidgets import interactive, interact
import ipywidgets as widgets
!pip install ipydatagrid >& /dev/null

!pip install healpy >& /dev/null
import healpy as hp
from healpy.newvisufunc import projview, newprojplot

!pip install geopandas >& /dev/null
import geopandas

!pip install geodatasets >& /dev/null
import geodatasets


import warnings
warnings.simplefilter("ignore")

from google.colab import output
output.enable_custom_widget_manager()
print("Tutte le librerie sono state correttamente importate")



---
# Dati
---

**Scarichiamo i dati da un server della NASA, potrebbe volerci qualche minuto, non preoccupatevi!**


In [ ]:
## Download data

all_ft2 = ["FERMI_POINTING_FINAL_044_2009092_2009099_00.fits",
           "FERMI_POINTING_FINAL_100_2010119_2010126_00.fits",
           "FERMI_POINTING_FINAL_150_2011104_2011111_00.fits",
           "FERMI_POINTING_FINAL_200_2012089_2012096_00.fits",
           "FERMI_POINTING_FINAL_250_2013073_2013080_00.fits",
           "FERMI_POINTING_FINAL_300_2014058_2014065_00.fits",
           "FERMI_POINTING_FINAL_350_2015043_2015050_00.fits",
           "FERMI_POINTING_FINAL_400_2016028_2016035_00.fits",
           "FERMI_POINTING_FINAL_450_2017012_2017019_01.fits",
           "FERMI_POINTING_FINAL_500_2017362_2018004_00.fits",
           "FERMI_POINTING_FINAL_550_2018347_2018354_00.fits",
           "FERMI_POINTING_FINAL_604_2019360_2020002_00.fits",
           "FERMI_POINTING_FINAL_654_2020345_2020352_00.fits",
           "FERMI_POINTING_FINAL_704_2021329_2021336_00.fits",
           "FERMI_POINTING_FINAL_772_2023075_2023082_00.fits",
           "FERMI_POINTING_FINAL_823_2024067_2024074_00.fits"
           #,"FERMI_POINTING_FINAL_873_2025051_2025058_00.fits"
           ]

spacecraft="Fermi_GLAST.png"
allskymapHpx = "Allsky_map_HPX.fits"

!wget https://fermi.gsfc.nasa.gov/ssc/observations/timeline/ft2/files/FERMI_POINTING_FINAL_044_2009092_2009099_00.fits -nc >& /dev/null
!wget https://fermi.gsfc.nasa.gov/ssc/observations/timeline/ft2/files/FERMI_POINTING_FINAL_100_2010119_2010126_00.fits -nc >& /dev/null
!wget https://fermi.gsfc.nasa.gov/ssc/observations/timeline/ft2/files/FERMI_POINTING_FINAL_150_2011104_2011111_00.fits -nc >& /dev/null
!wget https://fermi.gsfc.nasa.gov/ssc/observations/timeline/ft2/files/FERMI_POINTING_FINAL_200_2012089_2012096_00.fits -nc >& /dev/null
!wget https://fermi.gsfc.nasa.gov/ssc/observations/timeline/ft2/files/FERMI_POINTING_FINAL_250_2013073_2013080_00.fits -nc >& /dev/null
!wget https://fermi.gsfc.nasa.gov/ssc/observations/timeline/ft2/files/FERMI_POINTING_FINAL_300_2014058_2014065_00.fits -nc >& /dev/null
!wget https://fermi.gsfc.nasa.gov/ssc/observations/timeline/ft2/files/FERMI_POINTING_FINAL_350_2015043_2015050_00.fits -nc >& /dev/null
!wget https://fermi.gsfc.nasa.gov/ssc/observations/timeline/ft2/files/FERMI_POINTING_FINAL_400_2016028_2016035_00.fits -nc >& /dev/null
!wget https://fermi.gsfc.nasa.gov/ssc/observations/timeline/ft2/files/FERMI_POINTING_FINAL_450_2017012_2017019_01.fits -nc >& /dev/null
!wget https://fermi.gsfc.nasa.gov/ssc/observations/timeline/ft2/files/FERMI_POINTING_FINAL_500_2017362_2018004_00.fits -nc >& /dev/null
!wget https://fermi.gsfc.nasa.gov/ssc/observations/timeline/ft2/files/FERMI_POINTING_FINAL_550_2018347_2018354_00.fits -nc >& /dev/null
!wget https://fermi.gsfc.nasa.gov/ssc/observations/timeline/ft2/files/FERMI_POINTING_FINAL_604_2019360_2020002_00.fits -nc >& /dev/null
!wget https://fermi.gsfc.nasa.gov/ssc/observations/timeline/ft2/files/FERMI_POINTING_FINAL_654_2020345_2020352_00.fits -nc >& /dev/null
!wget https://fermi.gsfc.nasa.gov/ssc/observations/timeline/ft2/files/FERMI_POINTING_FINAL_704_2021329_2021336_00.fits -nc >& /dev/null
!wget https://fermi.gsfc.nasa.gov/ssc/observations/timeline/ft2/files/FERMI_POINTING_FINAL_772_2023075_2023082_00.fits -nc >& /dev/null
!wget https://fermi.gsfc.nasa.gov/ssc/observations/timeline/ft2/files/FERMI_POINTING_FINAL_823_2024067_2024074_00.fits -nc >& /dev/null
!wget https://fermi.gsfc.nasa.gov/ssc/observations/timeline/ft2/files/FERMI_POINTING_FINAL_873_2025051_2025058_00.fits -nc >& /dev/null
!wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=16wQAir7NTEHBaNU2tRXkFdwrYQt7LG46' -O $spacecraft >& /dev/null
!wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1Cyxr788w5N8xzXSbwvSP-EdydUCy4wBX' -O $allskymapHpx  >& /dev/null


print("Il contenuto della cartella è:")
!ls -lrt


---
Apriamo il file *FT2* e carichiamo i dati. Si tratta di una tabella che contiene le coordinate del satellite al variare del tempo.
Ecco alcune delle informazioni contenute nella tabella:

*   Tempo: definito come tempo di inizio (**START**) e tempo di fine (**STOP**)
*   Posizione del satellite sulla Terra: [latitudine](https://it.wikipedia.org/wiki/Latitudine) (**LAT_GEO**) e [longitudine](https://it.wikipedia.org/wiki/Longitudine) (**LON_GEO**)
*   Direzione del cielo al quale il satellite punta, identificata tramite un sistema di coordinate, dette coordinate celesti: [Ascensione retta](https://it.wikipedia.org/wiki/Ascensione_retta) (**RA_SCZ**) e [Declinazione](https://it.wikipedia.org/wiki/Declinazione_(astronomia)) (**DEC_SCZ**)
*   Posizione del Sole nel cielo, identificata sempre con coordinate celesti: **RA_SUN** e **DEC_SUN**

Abbiamo scaricato due file corrispondenti a due periodi di osservazione diversi. Possiamo commentare la prima riga (aggiungendo un '#' all'inizio) e scommentare la seconda (eliminando il '#') per vedere come cambiano i grafici successivi. Una volta fatta questa modifica è necessario eseguire nuovamente tutti i blocchi successivi per notare le differenze.




In [ ]:
# Apriamo il file
#filename_ft2 = "FERMI_POINTING_FINAL_044_2009092_2009099_00.fits" ## Apr '09
#filename_ft2 = "FERMI_POINTING_FINAL_604_2019360_2020002_00.fits" ## Dec '19
#filename_ft2 = "FERMI_POINTING_FINAL_772_2023075_2023082_00.fits" ## Mar '23
filename_ft2 = "FERMI_POINTING_FINAL_823_2024067_2024074_00.fits" ## Mar '24

hdu_ft2=pyfits.open(filename_ft2)
# Stampiamo le informazioni generali del file
print("Il contenuto del file è:")
hdu_ft2.info()
# Carichiamo i dati nella tabella chiamata 'SC_DATA'
tab_ft2 = Table(hdu_ft2['SC_DATA'].data)
print()
# Stampiamo i nomi delle colonne contenute nella tabella
print("Le colonne nella tabella sono:")
print(columnize(tab_ft2.colnames))

# Selezioniamo alcune colonne e visualizziamo le prime righe della tabella
print("Dati:")
max_entries = min(100,len(tab_ft2))
#tab_ft2['START','STOP','LAT_GEO', 'LON_GEO', 'RAD_GEO','RA_SUN', 'DEC_SUN', 'IN_SAA'][:max_entries].show_in_notebook(display_length=50)

if "RA_SUN" in tab_ft2.colnames:
  col_list = ['START','STOP','LAT_GEO', 'LON_GEO', 'RAD_GEO','RA_SUN', 'DEC_SUN', 'IN_SAA']
else:
  col_list = ['START','STOP','LAT_GEO', 'LON_GEO', 'RAD_GEO', 'IN_SAA']
tab_ft2[col_list][:max_entries].show_in_notebook(display_length=50)

# Prova a cambiare i nomi delle colonne inseriti nella riga precedente, scegliendoli tra i nomi delle colonne visualizzati qui in basso.

---

# Il tempo per il satellite Fermi

Gli istanti di tempo sono codificati in MET = Mission Elapsed Time, ovvero il tempo trascorso dall'inizio della missione.
Si è fissata convenzionalmente l'inizio della missione al primo gennaio del 2001, molto prima della data di lancio del satellite, avvenuta nel giugno 2008.
In effetti la progettazione e la costruzione di Fermi sono iniziati ben prima del lancio!



In [ ]:
## Conversione dei tempi
met0 = datetime.datetime(2001,1,1,0,0,0)


def met2date(met_seconds):
  return met0 + met_seconds*datetime.timedelta(seconds=1)
def datetime_to_astropy_time(dt):
    return Time(dt,format='datetime')

## Leggiamo i tempi dal file FT2
start = tab_ft2['START']
stop = tab_ft2['STOP']
start_date = met2date(start)
stop_date = met2date(stop)
print("L'inizio del file FT2 è:",start_date[0])
print("La fine del file FT2 è :",stop_date[-1])


---
# Traiettoria del satellite Fermi

Il file FT2 contiene la posizione del satellite al variare del tempo. Tale informazione è contenuto nelle colonne **LON_GEO** e **LAT_GEO** che rappresentano la longitudine e la latitudine del satellite. Leggiamo tali dati e rappresentiamoli.

Potremo notare che il satellite segue una orbita equatoriale e che la sua latitudine è sempre compresa tra 25°S e 25°N.


In [ ]:
## Traiettoria del satellite
# Mappa interattiva
# Esegui questo blocco e poi prova a modificare la variabile tempo usando la barra interattiva sul grafico
print("Cambia la variabile tempo per visualizzare la traiettoria del satellite")
print("Alcuni valori da provare: 50, 100, 200, 500, 1000, 1500 espressi in minuti")
lat_geo = tab_ft2['LAT_GEO']
lon_geo = tab_ft2['LON_GEO']
in_saa = tab_ft2['IN_SAA']
max_entries=len(tab_ft2)-1
if geopandas.__version__ <"1.0":
  world = geopandas.read_file(geopandas.datasets.get_path('naturalearth_lowres'))
else:
  world = geopandas.read_file(geodatasets.get_path('naturalearth.land'))
spacecraft = 'Fermi_GLAST.png'
arr_img = plt.imread(spacecraft, format='png')

play = widgets.Play(value=50,min=0,max=100,step=1,interval=500,description="Press play",disabled=False)
intslider = widgets.IntSlider(min=0, max=max_entries, step=1, value=0, description="Tempo (min)",continuous_update=False)
def plot_world(t):
  fig,ax = plt.subplots(figsize=(10,8));
  imagebox = OffsetImage(arr_img, zoom=0.04)
  imagebox.image.axes = ax
  world.plot(ax=ax)
  xy = [lon_geo[t],lat_geo[t]]
  ab = AnnotationBbox(imagebox, xy,xybox=(4., 2.),xycoords='data',boxcoords="offset points",pad=0.5, frameon=False)
  ax.add_artist(ab)

  #xx = lon_geo[:t+1]
  #yy = lat_geo[:t+1]
  #idx = np.where(np.abs(np.diff(xx))>180)[0].tolist()
  #idx = [0]+idx+[len(xx)]
  #tmp_xx = xx
  #tmp_yy = yy
  #for i in range(len(idx)-1):
  #  tmp_xx=xx[idx[i]+1:idx[i+1]]
  #  tmp_yy=yy[idx[i]+1:idx[i+1]]
  #  ax.plot(tmp_xx,tmp_yy,'k--')
  mask = ~in_saa
  ax.scatter(lon_geo[:t][mask[:t]],lat_geo[:t][mask[:t]], marker='.',s=10, color='k')
  ax.scatter(lon_geo[:t][in_saa[:t]],lat_geo[:t][in_saa[:t]], marker='.',s=10, color='r')

  plt.show()

w = interactive(plot_world, t=intslider)
output = w.children[-1]
output.layout.height = '500px'
display(w)


### Cosa rappresenta la zona in rosso?

# Altitudine di Fermi
L'altitudine del satellite è circa costante.

In [ ]:
time = met2date((tab_ft2['START']+tab_ft2['STOP'])/2)
alt = tab_ft2['RAD_GEO']/1000. ## km
alt_media = alt.mean()
print("L'altitudine media è {0:.2f} km".format(alt_media))

play = widgets.Play(value=0,min=0,max=max_entries,step=1,interval=100,description="Press play",disabled=False)
intslider = widgets.IntSlider(min=0, max=max_entries, step=1, value=0,description="Tempo (min)", continuous_update=False)

def plot_altitude(i):
  fig,ax = plt.subplots(figsize=(20,4))
  ax.plot(time,alt, '-')
  ax.plot(time,[alt_media]*len(time), 'k--')
  ax.plot(time[i],alt[i], 'or')
  ax.set_title('Altitudine')
  ax.set_xlabel('Data')
  ax.set_ylabel('Altitudine (km)')
  ax.set_ylim(450,550)
  fig.autofmt_xdate()

  plt.show()

w = interactive(plot_altitude, i=intslider)
output = w.children[-1]
output.layout.height = '400px'
display(w)


Tuttavia, nell'arco della intera missione l'altitudine decresce nel tempo...

In [ ]:
all_time = []
all_alt = []
for _ft2 in all_ft2:
  tmp_hdu_ft2=pyfits.open(_ft2)
  tmp_tab_ft2 = Table(tmp_hdu_ft2['SC_DATA'].data)
  tmp_time = met2date((tmp_tab_ft2['START'][0]+tmp_tab_ft2['STOP'][-1])/2)
  tmp_alt = tmp_tab_ft2['RAD_GEO']/1000. ## km
  tmp_alt_media = tmp_alt.mean()

  all_time.append(tmp_time)
  all_alt.append(tmp_alt_media)

all_time = np.array(all_time)
all_alt = np.array(all_alt)


fig,ax = plt.subplots(figsize=(10,4))
ax.plot(all_time,all_alt, '.--')
ax.set_title('Altitudine - intera missione')
ax.set_xlabel('Data')
ax.set_ylabel('Altitudine media (km)')
fig.autofmt_xdate()
plt.show()


---
# Puntamento del satellite

Dove guarda il satellite Fermi?
Il satellite guarda una porzione di cielo per volta (indicata dal cerchio rosso), ma si muove velocemente nel cielo ed in circa 3 ore ha gaurdato quasi completamente tutto il cielo.

La porzione di cielo che il satellite osserva si chiama "*Field of view*" ed nel caso del Fermi-LAT è circa il 20% dell'intero cielo!



In [ ]:
## Puntamento del satellite
# Mappa interattiva
# Esegui questo blocco e poi prova a modificare la variabile tempo usando la barra interattiva sul grafico
print("La traiettoria tratteggiata indica come è cambiata la direzione di puntamento nel tempo.")
print("Il puntino rosso indica la direzione al tempo selezionato e la curva rossa rappresenta il Field-of-view.")
print("Cambiare la variabile tempo per visualizzare il puntamento del satellite")
print("Alcuni valori da provare: 30, 50, 70, 100, 1000, 1500")

## Leggiamo le coordinate di puntamento
ra_z = tab_ft2['RA_SCZ']
dec_z = tab_ft2['DEC_SCZ']
c = SkyCoord(ra=ra_z*u.degree, dec=dec_z*u.degree, frame='fk5').galactic
lat_z = c.b
lon_z = c.l
max_entries=len(tab_ft2)-1
lon_z = np.where(lon_z.deg>=180, lon_z-360*u.degree,lon_z)

## Calcoliamo il campo di vista, detto Field-of-view (FoV)
FoV=2.4 #sr
theta = np.arccos(1-FoV/(2*np.pi))*180/np.pi
m = hp.read_map(allskymapHpx, verbose=False);

play = widgets.Play(value=0,min=0,max=max_entries,step=1,interval=500,description="Press play",disabled=False)
intslider = widgets.IntSlider(min=0, max=max_entries, step=1, value=0,description="Tempo (min)", continuous_update=False)
def plot_sky(t):
  fig = plt.figure(figsize=(8,6))
  #'''
  hp.mollview(m, fig=fig.number, title='Il cielo gamma', norm=LogNorm(), badcolor='white', cbar=True);
  hp.graticule(verbose=False);
  hp.projplot(lon_z[:t], lat_z[:t], lonlat=True, color='k', ls='--');
  circle = SphericalCircle([lon_z[t],lat_z[t]],theta*u.deg, color='r')
  xy = circle.get_xy().T
  hp.projscatter(lon_z[t], lat_z[t], lonlat=True, color='r', marker='o');
  hp.projplot(xy[0],xy[1],lonlat=True,color='r');
  plt.show()

w = interactive(plot_sky, t=intslider)
output = w.children[-1]
#output.layout.height = '380px'
display(w)



---
# La posizione del Sole

Dove si trova il Sole?
Possiamo individuare la posizione del Sole in ogni istante e visualizzare la sua traiettoria nel cielo.



In [ ]:
## Puntamento del satellite
# Mappa interattiva
# Esegui questo blocco e poi prova a modificare la variabile tempo usando la barra interattiva sul grafico
print("Il puntino rosso indica la posizione del Sole in funzione del tempo.")
print("La traiettoria tratteggiata indica la traiettoria seguita dal Sole.")
print("Alcuni valori da provare: 30, 90, 180, 360")

# Definiamo l'intervallo in cui visualizzare la traiettoria del Sole
start_datetime = datetime.datetime(2020,1,1,0,0,0)
stop_datetime = datetime.datetime(2024,1,1,0,0,0)
start_time = datetime_to_astropy_time(start_datetime)
stop_time = datetime_to_astropy_time(stop_datetime)
time_step = datetime.timedelta(days=1)
nstep = int((stop_datetime-start_datetime)/time_step)
datetime_array = np.arange(start_datetime, stop_datetime, time_step).tolist()
time_array = datetime_to_astropy_time(datetime_array)


## Leggiamo le coordinate del Sole
sun_coord = get_sun(time_array)
c = SkyCoord(ra=sun_coord.ra.deg*u.degree, dec=sun_coord.dec.deg*u.degree, frame='fk5')
lat_sun = c.galactic.b
lon_sun = c.galactic.l


## Definiamo l'equatore celeste
c = SkyCoord(ra=np.arange(-180,180,1)*u.degree, dec=0*u.degree, frame='fk5')
lat_eq = c.galactic.b
lon_eq = c.galactic.l


m = hp.read_map(allskymapHpx, verbose=False);

play = widgets.Play(value=0,min=0,max=nstep,step=1,interval=500,description="Press play",disabled=False)
intslider = widgets.IntSlider(min=0, max=nstep, step=1, value=0,description="Tempo (days)", continuous_update=False)
def plot_sky(t):
  fig = plt.figure(figsize=(8,6))
  hp.mollview(m, fig=fig.number, title='La traiettoria del sole', norm=LogNorm(), badcolor='white');
  hp.graticule(verbose=False);
  hp.projplot(lon_sun[:t], lat_sun[:t], lonlat=True, color='orange', ls='--');
  hp.projscatter(lon_sun[t], lat_sun[t], lonlat=True, color='r', marker='*');
  hp.projplot(lon_eq, lat_eq, lonlat=True, color='white', ls='--');
  hp.projtext(35,15,"equatore celeste",lonlat=True,color='white', rotation=55)
  plt.show()

w = interactive(plot_sky, t=intslider)
output = w.children[-1]
#output.layout.height = '380px'
display(w)




---
# Tempo di osservazione

Nella modalità sky survey il satellite osserva l'intero cielo all'incirca ogni due orbite. Possiamo calcolare il tempo che il satellite "spende" ad osservare un dato pixel del cielo, ovvero ciò che viene definito "livetime".

In [ ]:
## Mappa di livetime
# Mappa interattiva
# Esegui questo blocco e poi prova a modificare la variabile tempo usando la barra interattiva sul grafico
print("Il puntino rosso indica la direzione al tempo selezionato e la curva rossa rappresenta il Field-of-view.")
print("Cambiare la variabile tempo per visualizzare l'incremento del tempo di osservazione.")
print("Alcuni valori da provare: 30, 50, 70, 100, 1000, 1500")

## Leggiamo le coordinate di puntamento
ra_z = tab_ft2['RA_SCZ']
dec_z = tab_ft2['DEC_SCZ']
c = SkyCoord(ra=ra_z*u.degree, dec=dec_z*u.degree, frame='fk5').galactic
lat_z = c.b
lon_z = c.l
max_entries=len(tab_ft2)-1
lon_z = np.where(lon_z.deg>=180, lon_z-360*u.degree,lon_z)

## Calcoliamo il campo di vista, detto Field-of-view (FoV)
FoV=2.4 #sr
theta = np.arccos(1-FoV/(2*np.pi))*180/np.pi
m = hp.read_map(allskymapHpx, verbose=False);
m1 = np.ones(len(m))

play = widgets.Play(value=1,min=1,max=max_entries+1,step=1,interval=500,description="Press play",disabled=False)
intslider = widgets.IntSlider(min=1, max=max_entries+1, step=1, value=1,description="Tempo (min)", continuous_update=False)
def plot_sky(t):
  t-=1
  m1 = np.zeros(len(m))
  for _t in range(t+1):
    vec = hp.pixelfunc.ang2vec(lon_z[_t].deg,lat_z[_t].deg, lonlat=True)
    idx_qd = hp.query_disc(hp.npix2nside(len(m1)), vec, theta/180*np.pi, inclusive=False)
    m1[idx_qd] += 1

  fig = plt.figure(figsize=(8,6))
  #'''
  hp.mollview(m1, fig=fig.number, title='Tempo di osservazione', badcolor='white', cbar=True, unit='Livetime (min)',);
  hp.graticule(verbose=False);
  #hp.projplot(lon_z[:t], lat_z[:t], lonlat=True, color='k', ls='--');
  circle = SphericalCircle([lon_z[t],lat_z[t]],theta*u.deg, color='r')
  xy = circle.get_xy().T
  hp.projscatter(lon_z[t], lat_z[t], lonlat=True, color='r', marker='o');
  hp.projplot(xy[0],xy[1],lonlat=True,color='r');
  plt.show()

w = interactive(plot_sky, t=intslider)
output = w.children[-1]
#output.layout.height = '380px'
display(w)


# THE END